In [1]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 744.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 85.8 MB/s eta 0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0


In [3]:
from google.colab import drive
import pickle
import numpy as np
import os

drive.mount('/content/drive')

def load_batch(file_path):
    with open(file_path, 'rb') as f:
        dict_data = pickle.load(f, encoding='bytes')
        features = dict_data[b'data']
        labels = dict_data[b'labels']
        features = features.reshape((len(features), 3, 32, 32)).transpose(0, 2, 3, 1)
        return features, labels

def load_full_dataset(path):
    x_train, y_train = [], []
    for i in range(1, 6):
        batch_path = os.path.join(path, f'data_batch_{i}')
        features, labels = load_batch(batch_path)
        x_train.append(features)
        y_train.extend(labels)

    x_train = np.concatenate(x_train)
    y_train = np.array(y_train).reshape(-1, 1)

    test_batch_path = os.path.join(path, 'test_batch')
    x_test, y_test = load_batch(test_batch_path)
    x_test = x_test
    y_test = np.array(y_test).reshape(-1, 1)

    return (x_train, y_train), (x_test, y_test)

dataset_path = '/content/drive/MyDrive/cifar-10-batches-py'

(x_train, y_train), (x_test, y_test) = load_full_dataset(dataset_path)

x_train = x_train / 255.0
x_test = x_test / 255.0

print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

Mounted at /content/drive
(50000, 32, 32, 3) (50000, 1)
(10000, 32, 32, 3) (10000, 1)


In [9]:
import matplotlib.pyplot as plt
import numpy as np

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_train[i])
    # Make subplot x-labels bold
    plt.xlabel(class_names[int(y_train[i][0])], fontweight='bold')
plt.suptitle('Sample Images from CIFAR-10', fontweight='bold') # Add a bold suptitle for the entire figure
plt.savefig('sample_images.eps', format='eps', dpi=600, bbox_inches='tight')
plt.close()

unique, counts = np.unique(y_train, return_counts=True)
plt.figure(figsize=(8, 4))
plt.bar(class_names, counts)
plt.title('Class Distribution in CIFAR-10', fontweight='bold')
plt.xlabel('Classes', fontweight='bold')
plt.ylabel('Frequency', fontweight='bold')
plt.xticks(rotation=45, fontweight='bold') # Make x-tick labels bold
plt.yticks(fontweight='bold') # Make y-tick labels bold
plt.savefig('class_distribution.eps', format='eps', dpi=600, bbox_inches='tight')
plt.close()

In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models

def compute_spatial_dimensions(kernel_size, stride, padding_type):
    model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(16, kernel_size, strides=stride, padding=padding_type)
    ])
    return model.output_shape

configurations = [
    ((3, 3), 1, 'same'), ((5, 5), 1, 'same'), ((7, 7), 1, 'same'),
    ((3, 3), 1, 'valid'), ((3, 3), 2, 'same'), ((3, 3), 2, 'valid')
]

for k, s, p in configurations:
    shape = compute_spatial_dimensions(k, s, p)
    print(f"Kernel: {k}, Stride: {s}, Padding: {p} -> Output Shape: {shape}")

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Kernel: (3, 3), Stride: 1, Padding: same -> Output Shape: (None, 32, 32, 16)
Kernel: (5, 5), Stride: 1, Padding: same -> Output Shape: (None, 32, 32, 16)
Kernel: (7, 7), Stride: 1, Padding: same -> Output Shape: (None, 32, 32, 16)
Kernel: (3, 3), Stride: 1, Padding: valid -> Output Shape: (None, 30, 30, 16)
Kernel: (3, 3), Stride: 2, Padding: same -> Output Shape: (None, 16, 16, 16)
Kernel: (3, 3), Stride: 2, Padding: valid -> Output Shape: (None, 15, 15, 16)


In [6]:
def build_and_compile_cnn(pooling_layer):
    model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        pooling_layer((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        pooling_layer((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

model_max = build_and_compile_cnn(layers.MaxPooling2D)
model_avg = build_and_compile_cnn(layers.AveragePooling2D)

print("Training CNN with Max Pooling...")
history_max = model_max.fit(x_train, y_train, epochs=20, batch_size=32, validation_data=(x_test, y_test), verbose=1)

print("\nTraining CNN with Average Pooling...")
history_avg = model_avg.fit(x_train, y_train, epochs=20, batch_size=32, validation_data=(x_test, y_test), verbose=1)

Training CNN with Max Pooling...
Epoch 1/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - accuracy: 0.4641 - loss: 1.4900 - val_accuracy: 0.5793 - val_loss: 1.1931
Epoch 2/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - accuracy: 0.6064 - loss: 1.1264 - val_accuracy: 0.6425 - val_loss: 1.0402
Epoch 3/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - accuracy: 0.6528 - loss: 0.9948 - val_accuracy: 0.6538 - val_loss: 1.0090
Epoch 4/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - accuracy: 0.6813 - loss: 0.9174 - val_accuracy: 0.6505 - val_loss: 1.0004
Epoch 5/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - accuracy: 0.7013 - loss: 0.8532 - val_accuracy: 0.6606 - val_loss: 1.0052
Epoch 6/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - accuracy: 0.7216 - loss: 0.8027 - val_accuracy: 0.6832 - val_loss: 0.9204
Epoch 7/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - accuracy: 0.7383 - loss: 0.7504 - val_accuracy: 0.6874 - val_loss: 0.9259
Epoch 8/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 2

In [10]:
import matplotlib.pyplot as plt
import tensorflow as tf

first_conv_layer = model_max.layers[0]
activations = first_conv_layer(x_test[:1])
first_layer_activation = activations.numpy()

fig = plt.figure(figsize=(12, 6))
for i in range(8):
    ax = fig.add_subplot(2, 4, i+1)
    ax.matshow(first_layer_activation[0, :, :, i], cmap='viridis')
    ax.axis('off')
    # Make tick labels bold (though they are off, this sets the property)
    for tick in ax.get_xticklabels():
        tick.set_fontweight('bold')
    for tick in ax.get_yticklabels():
        tick.set_fontweight('bold')

plt.suptitle('Feature Maps from the First Convolutional Layer', fontweight='bold') # Add a bold suptitle
plt.savefig('feature_maps.eps', format='eps', dpi=600, bbox_inches='tight')
plt.close()

In [8]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

plt.figure(figsize=(8, 4))
plt.plot(history_max.history['accuracy'], label='Training Accuracy')
plt.plot(history_max.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy', fontweight='bold')
plt.xlabel('Epoch', fontweight='bold')
plt.ylabel('Accuracy', fontweight='bold')
plt.legend(loc='lower right', prop={'weight': 'bold'})
plt.savefig('training_validation_accuracy.eps', format='eps', dpi=600, bbox_inches='tight')
plt.close()

plt.figure(figsize=(8, 4))
plt.plot(history_max.history['loss'], label='Training Loss')
plt.plot(history_max.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss', fontweight='bold')
plt.xlabel('Epoch', fontweight='bold')
plt.ylabel('Loss', fontweight='bold')
plt.legend(loc='upper right', prop={'weight': 'bold'})
plt.savefig('training_validation_loss.eps', format='eps', dpi=600, bbox_inches='tight')
plt.close()

y_pred = model_max.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_classes, target_names=class_names))

cm = confusion_matrix(y_test, y_pred_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(10, 10))
disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation=45)

# Make x and y axis labels bold for confusion matrix
ax.set_xlabel(ax.get_xlabel(), fontweight='bold')
ax.set_ylabel(ax.get_ylabel(), fontweight='bold')

# Make tick labels bold
for tick in ax.get_xticklabels():
    tick.set_fontweight('bold')
for tick in ax.get_yticklabels():
    tick.set_fontweight('bold')

plt.savefig('confusion_matrix.eps', format='eps', dpi=600, bbox_inches='tight')
plt.close()

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
              precision    recall  f1-score   support

    airplane       0.75      0.70      0.72      1000
  automobile       0.79      0.79      0.79      1000
        bird       0.58      0.61      0.59      1000
         cat       0.49      0.41      0.45      1000
        deer       0.66      0.57      0.61      1000
         dog       0.52      0.67      0.59      1000
        frog       0.72      0.79      0.75      1000
       horse       0.77      0.68      0.73      1000
        ship       0.80      0.78      0.79      1000
       truck       0.76      0.79      0.77      1000

    accuracy                           0.68     10000
   macro avg       0.68      0.68      0.68     10000
weighted avg       0.68      0.68      0.68     10000

